## Q: Zigzag Conversionnies
The string "PAYPALISHIRING" is written in a zigzag pattern on a given number of rows like this: (you may want to display this pattern in a fixed font for better legibility)

P   A <br>  H   N
A P L <br> S I I G
 <br>Y   I   R
And then read line by line: "PAHNAPLSIIGYIR"

Write the code that will take a string and make this conversion given a number of rows:

string convert(string s, int numRows);
 

# Conventional Solution
The simulation approach solves the problem by mimicking the exact process of writing the string in a zigzag pattern row by row.

We create a separate container for each row and iterate through the characters of the input string one at a time. As we process each character, we place it into the current row. We then move either downward or upward through the rows depending on our current direction.

In [ ]:
class Solution:
    def convert(self, s: str, numRows: int) -> str:
        if numRows == 1 or numRows >= len(s):
            return s


        rows = [""] * numRows


        current_row = 0
        going_down = False

        for char in s:

            rows[current_row] += char

            if current_row == 0 or current_row == numRows - 1:
                going_down = not going_down

            if going_down:
                current_row += 1
            else:
                current_row -= 1

        
        return "".join(rows)

## Optimised Mathematical Approach

The optimized approach avoids explicitly simulating the zigzag movement. Instead, it relies on recognizing a repeating mathematical pattern in the character indices.

The key observation is that the zigzag structure repeats in cycles. For a zigzag with numRows, one complete cycle contains:

cycle_len = 2 * numRows - 2

characters.

Using this cycle length, we can directly determine which characters belong to each row without constructing the zigzag visually.

The first row contains characters spaced exactly one cycle apart.
The last row also follows the same rule.
The middle rows are special because each cycle contributes two characters:
1. a vertical character
2. a diagonal character

For each row, we iterate through the string in jumps of cycle_len. For middle rows, we additionally calculate the diagonal character index using:

diagonal_index = current_index + cycle_len - 2 * row

This allows us to build the final result directly in row order.

In [11]:
class Solution:
    def convert(self, s: str, numRows: int) -> str:
        if numRows == 1 or numRows >= len(s):
            return s

        result = []


        cycle_len = 2 * numRows - 2


        for row in range(numRows):

            for j in range(row, len(s), cycle_len):
                result.append(s[j])

                diagonal_index = j + cycle_len - 2 * row
                if (
                    row != 0
                    and row != numRows - 1
                    and diagonal_index < len(s)
                ):
                    result.append(s[diagonal_index])

        return "".join(result)

## Q: Reverse unsigned integers

Given a signed 32-bit integer x, return x with its digits reversed. If reversing x causes the value to go outside the signed 32-bit integer range [-2<sup>31</sup>, 2<sup>31</sup> - 1], then return 0.

Assume the environment does not allow you to store 64-bit integers (signed or unsigned).

## Approach 1: Mathematical Digit Extraction
Instead of forming the full reversed number and checking afterwards, we build it one digit at a time using modulo and floor division. Critically, we check for overflow before each multiplication step. This respects the constraint of never actually holding a value outside the 32-bit range.
The key insight is that rev * 10 + digit > INT_MAX can be rearranged to rev > (INT_MAX - digit) // 10, which lets us detect the violation using only values already within bounds. 

Working on the absolute value throughout and reapply the sign at the end, this simplifies the overflow logic to a single direction (against INT_MAX), which is safe because |INT_MIN| > INT_MAX, so anything fitting in INT_MAX will always fit in the negative range too.

In [ ]:
class Solution:
    def reverse(self, x: int) -> int:
        INT_MAX = 2**31 - 1

        sign = -1 if x < 0 else 1
        x = abs(x)
        rev = 0

        while x:
            digit = x % 10
            x //= 10

            if rev > (INT_MAX - digit) // 10:
                return 0

            rev = rev * 10 + digit

        return sign * rev

## Approach 2: String Manipulation

The mental model: strip the sign, reverse the digit characters as a string, reattach the sign, then do a bounds check at the end. It's intuitive but leans on Python's arbitrary-precision integers under the hood.

In [ ]:
class Solution:
    def reverse(self, x: int) -> int:
        INT_MIN, INT_MAX = -(2**31), 2**31 - 1

        sign = -1 if x < 0 else 1
        result = sign * int(str(abs(x))[::-1])

        return result if INT_MIN <= result <= INT_MAX else 0

## Q: Next Permutation
A permutation of an array of integers is an arrangement of its members into a sequence or linear order.

For example, for arr = [1,2,3], the following are all the permutations of arr: [1,2,3], [1,3,2], [2, 1, 3], [2, 3, 1], [3,1,2], [3,2,1].
The next permutation of an array of integers is the next lexicographically greater permutation of its integer. More formally, if all the permutations of the array are sorted in one container according to their lexicographical order, then the next permutation of that array is the permutation that follows it in the sorted container. If such arrangement is not possible, the array must be rearranged as the lowest possible order (i.e., sorted in ascending order).

For example, the next permutation of arr = [1,2,3] is [1,3,2].
Similarly, the next permutation of arr = [2,3,1] is [3,1,2].
While the next permutation of arr = [3,2,1] is [1,2,3] because [3,2,1] does not have a lexicographical larger rearrangement.
Given an array of integers nums, find the next permutation of nums.

The replacement must be in place and use only constant extra memory.

## Conventional Approach: Sort the Suffix
The idea here follows the same core logic as the optimised version, but takes a shortcut on the last step. After finding where the array "breaks" and doing the swap, we just sort the remaining suffix rather than thinking about why a reverse would suffice. It works, but it throws away information we actually already have, costing us O(n log n) on that final step.
Start from the right and find the first index i where the value dips below its right neighbour. That's the pivot. Then find the smallest value to the right of i that's still larger than nums[i], swap them, and sort everything to the right of i in ascending order. If no pivot exists, the whole array is in descending order, so just sort the entire thing.

In [ ]:
class Solution:
    def nextPermutation(self, nums: list[int]) -> None:
        n = len(nums)
        pivot = -1

        for i in range(n - 2, -1, -1):
            if nums[i] < nums[i + 1]:
                pivot = i
                break

        if pivot == -1:
            nums.sort()
            return

        for j in range(n - 1, pivot, -1):
            if nums[j] > nums[pivot]:
                nums[pivot], nums[j] = nums[j], nums[pivot]
                break

        nums[pivot + 1:] = sorted(nums[pivot + 1:])

## Optimised Approach: Reverse the Suffix
The key realisation is that once the pivod is found and the swap done, everything to the right of the pivot is guaranteed to still be in descending order. So instead of sorting it, you just reverse it to get ascending order in O(n). That one observation is what separates this solution from the conventional one.


Walk right to left to find the first index i where the sequence stops being non-increasing. That's your pivot, and nums[i] is the digit that needs to get bigger. Then walk right to left again and find the first value larger than nums[i], swap them. Now the suffix is still descending (swapping with the rightmost valid candidate preserves this), so a simple two-pointer reverse brings it to ascending order and gives the lexicographically smallest continuation.

In [ ]:
class Solution:
    def nextPermutation(self, nums: list[int]) -> None:
        n = len(nums)
        pivot = -1

        for i in range(n - 2, -1, -1):
            if nums[i] < nums[i + 1]:
                pivot = i
                break

        if pivot == -1:
            nums.reverse()
            return

        for j in range(n - 1, pivot, -1):
            if nums[j] > nums[pivot]:
                nums[pivot], nums[j] = nums[j], nums[pivot]
                break

        left, right = pivot + 1, n - 1
        while left < right:
            nums[left], nums[right] = nums[right], nums[left]
            left += 1
            right -= 1

## Q: Container with most water
You are given an integer array height of length n. There are n vertical lines drawn such that the two endpoints of the ith line are (i, 0) and (i, height[i]).

Find two lines that together with the x-axis form a container, such that the container contains the most water.

Return the maximum amount of water a container can store.

## Approach: Two Pointers
This approach exploits a simple but powerful observation: start with the widest possible container (left = 0, right = n-1). From here, the only way to find more water is to find a taller bounding line, since the width can only shrink as we move inward. So we always discard the shorter of the two current lines by moving that pointer inward. There is no point keeping a short line around since it is already the bottleneck, and any future pairing with it will have less width on top of the same height ceiling.


This way every element is visited at most once, and we never miss the optimal pair because we only ever discard lines that could not possibly improve the answer.

In [6]:
class Solution:
    def maxArea(self, height: list[int]) -> int:
        left, right = 0, len(height) - 1
        max_water = 0

        while left < right:
            water = min(height[left], height[right]) * (right - left)
            max_water = max(max_water, water)

            if height[left] < height[right]:
                left += 1
            else:
                right -= 1

        return max_water

## Q: Integer to Roman numerals
Roman numerals are formed by appending the conversions of decimal place values from highest to lowest. Given an integer, convert it to a Roman numeral.

## Conventional Approach: Greedy Subtraction
This follows the problem description almost literally. Keep a list of all meaningful values (including the subtractive forms like 4, 9, 40, etc.) in descending order. Then repeatedly find the largest value that fits into the remaining number, append its symbol, and subtract it. Keep going until nothing remains. It reads naturally but involves a repeated linear scan through the values list on each iteration.

In [ ]:
class Solution:
    def intToRoman(self, num: int) -> str:
        values = [1000,900,500,400,100,90,50,40,10,9,5,4,1]
        symbols = ["M","CM","D","CD","C","XC","L","XL","X","IX","V","IV","I"]

        result = []

        for i, val in enumerate(values):
            while num >= val:
                result.append(symbols[i])
                num -= val

        return "".join(result)

## Optimised Approach: Digit Decomposition

Rather than repeatedly subtracting and scanning, we decompose the number into its decimal place values upfront: 1000, 100, 10 and 1. Each place has exactly 13 possible values (0 through 9, but with the Roman encoding already worked out), so we just index directly into four lookup tables. No loops beyond the final join, no repeated scanning, just four direct lookups and done.


The tables encode every possible digit value for each place, so the subtractive rules are baked in rather than computed on the fly. It trades a bit more memory for a cleaner, branchless resolution.

In [ ]:
class Solution:
    def intToRoman(self, num: int) -> str:
        thousands = ["", "M", "MM", "MMM"]
        hundreds  = ["", "C", "CC", "CCC", "CD", "D", "DC", "DCC", "DCCC", "CM"]
        tens      = ["", "X", "XX", "XXX", "XL", "L", "LX", "LXX", "LXXX", "XC"]
        ones      = ["", "I", "II", "III", "IV", "V", "VI", "VII", "VIII", "IX"]

        return (
            thousands[num // 1000] +
            hundreds[(num % 1000) // 100] +
            tens[(num % 100) // 10] +
            ones[num % 10]
        )

## Q: 4Sum

Given an array nums of n integers, return an array of all the unique quadruplets [nums[a], nums[b], nums[c], nums[d]] such that:

0 <= a, b, c, d < n
a, b, c, and d are distin ct.
nums[a] + nums[b] + nums[c] + nums[d] == target

## Conventional Approach: Nested Loops

Simply create three indices with nested loops, then check if the fourth required value exists in a set built from the remaining elements. Using a set gives O(1) lookup, which saves one full loop compared to pure brute force. Deduplication is handled by storing results as sorted tuples in a set before converting to a list at the end. It works but the triple loop will be slow on large inputs.

In [ ]:
class Solution:
    def fourSum(self, nums: list[int], target: int) -> list[list[int]]:
        nums.sort()
        n = len(nums)
        results = set()

        for i in range(n - 2):
            for j in range(i + 1, n - 1):
                seen = set()
                for k in range(j + 1, n):
                    need = target - nums[i] - nums[j] - nums[k]
                    if need in seen:
                        results.add((nums[i], nums[j], need, nums[k]))
                    seen.add(nums[k])

        return [list(q) for q in results]

## Optimised Approach: Sort first then apply 2 pointers
Sort the array first, then fix the first two indices with two outer loops. For each pair, run a two pointer sweep on the remaining subarray to find pairs that complete the target. Sorting is what makes the two pointer logic valid, and it also makes deduplication cheap: whenever a pointer lands on the same value it just saw, skip it. No sets, no hash lookups, just pointer movement.


The duplicate skipping is worth paying attention to. After fixing index i, if nums[i] = nums[i-1], we already explored that branch, so skip. Same for j. Inside the two pointer loop, after a valid quad is found and the pointers move, skip over any repeated values on both ends before the next iteration. This keeps the result set unique without any extra data structures.

In [ ]:
class Solution:
    def fourSum(self, nums: list[int], target: int) -> list[list[int]]:
        nums.sort()
        n = len(nums)
        results = []

        for i in range(n - 3):
            if i > 0 and nums[i] == nums[i - 1]:
                continue

            for j in range(i + 1, n - 2):
                if j > i + 1 and nums[j] == nums[j - 1]:
                    continue #ensure within bound

                left, right = j + 1, n - 1

                while left < right:
                    total = nums[i] + nums[j] + nums[left] + nums[right]

                    if total == target:
                        results.append([nums[i], nums[j], nums[left], nums[right]])
                        while left < right and nums[left] == nums[left + 1]: #check duplicates
                            left += 1
                        while left < right and nums[right] == nums[right - 1]: #check duplicates
                            right -= 1
                        left += 1
                        right -= 1
                    elif total < target:
                        left += 1
                    else:
                        right -= 1

        return results

## Q: Divide Two IntegersGiven two integers dividend and divisor, divide two integers without using multiplication, division, and mod operator.

The integer division should truncate toward zero, which means losing its fractional part. For example, 8.345 would be truncated to 8, and -2.7335 would be truncated to -2.

Return the quotient after dividing dividend by divisoNote: Assume we are dealing with an environment that could only store integers within the 32-bit signed integer range:[-2^31, 2^31-1]. For this problem, if the quotient is strictly greater than 2^31 - 1, then return 2^31 - 1, and if the quotient is strictly less than -2^31, then return it -231.


## Conventional Approach: Keep doing subtraction
Keep subtracting the divisor and counting how many times we do it. Handle the sign separately by working with absolute values throughout, then reapply at the end.

In [ ]:
class Solution:
    def divide(self, dividend: int, divisor: int) -> int:
        INT_MAX = 2**31 - 1
        INT_MIN = -(2**31)

        if dividend == INT_MIN and divisor == -1:
            return INT_MAX

        negative = (dividend < 0) != (divisor < 0)
        a = abs(dividend)
        b = abs(divisor)

        count = 0
        while a >= b:
            a -= b
            count += 1

        return -count if negative else count

## Optimised Approach: Left bit shifts. 

In binary notation, shifting a number left by 1 is the same as doubling it, and we can do this without multiplication. So rather than subtracting b once each iteration, we find the largest b * 2^n that still fits inside a, subtract that whole chunk, and credit 2^n to our count. Then repeat on the remainder.

In [53]:
class Solution:
    def divide(self, dividend: int, divisor: int) -> int:
        INT_MAX = 2**31 - 1
        INT_MIN = -(2**31)

        if dividend == INT_MIN and divisor == -1:
            return INT_MAX

        negative = (dividend < 0) != (divisor < 0)
        a = abs(dividend)
        b = abs(divisor)

        count = 0
        while a >= b:
            shifted = b
            multiple = 1

            while a >= (shifted << 1):
                shifted <<= 1
                multiple <<= 1

            a -= shifted
            count += multiple

        return -count if negative else count

## Q: Determine if the Sudoku Biard is valid 

Only the filled cells need to be validated according to the following rules:

Each row must contain the digits 1-9 without repetition.
Each column must contain the digits 1-9 without repetition.
Each of the nine 3 x 3 sub-boxes of the grid must contain the digits 1-9 without repetition.

## Conventional Approach: Brute force checking
Make three separate passes, one for rows, one for columns, one for boxes. Each pass collects digits and checks for duplicates.

In [ ]:
class Solution:
    def isValidSudoku(self, board: list[list[str]]) -> bool:
        for i in range(9):
            row = [x for x in board[i] if x != '.']
            if len(row) != len(set(row)):
                return False

        for c in range(9):
            col = [board[r][c] for r in range(9) if board[r][c] != '.']
            if len(col) != len(set(col)):
                return False

        for box_r in range(3):
            for box_c in range(3):
                box = [
                    board[r][c]
                    for r in range(box_r * 3, box_r * 3 + 3)
                    for c in range(box_c * 3, box_c * 3 + 3)
                    if board[r][c] != '.'
                ]
                if len(box) != len(set(box)):
                    return False

        return True

## Optimised Approach: Single Pass
Do one pass for the entire board. For every cell, record the digit in all three trackers simultaneously. If any digit has already been seen in that row, column, or box, return False immediately without finishing the scan.

In [ ]:
class Solution:
    def isValidSudoku(self, board: list[list[str]]) -> bool:
        rows = [set() for _ in range(9)]
        cols = [set() for _ in range(9)]
        boxes = [set() for _ in range(9)]

        for r in range(9):
            for c in range(9):
                val = board[r][c]
                if val == '.':
                    continue

                box_idx = (r // 3) * 3 + (c // 3)

                if val in rows[r] or val in cols[c] or val in boxes[box_idx]:
                    return False

                rows[r].add(val)
                cols[c].add(val)
                boxes[box_idx].add(val)

        return True